In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("SakilaAnalysis") \
    .getOrCreate()

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [4]:
from google.colab import files
uploaded = files.upload()

Saving film_actor.csv to film_actor.csv
Saving inventory.csv to inventory.csv
Saving city.csv to city.csv
Saving actor.csv to actor.csv
Saving payment.csv to payment.csv
Saving category.csv to category.csv
Saving address.csv to address.csv
Saving film_category.csv to film_category.csv
Saving customer.csv to customer.csv
Saving film.csv to film.csv
Saving rental.csv to rental.csv


In [5]:
film = spark.read.csv("film.csv", header=True, inferSchema=True)
category = spark.read.csv("category.csv", header=True, inferSchema=True)
film_category = spark.read.csv("film_category.csv", header=True, inferSchema=True)
actor = spark.read.csv("actor.csv", header=True, inferSchema=True)
film_actor = spark.read.csv("film_actor.csv", header=True, inferSchema=True)
inventory = spark.read.csv("inventory.csv", header=True, inferSchema=True)
rental = spark.read.csv("rental.csv", header=True, inferSchema=True)
payment = spark.read.csv("payment.csv", header=True, inferSchema=True)
customer = spark.read.csv("customer.csv", header=True, inferSchema=True)
address = spark.read.csv("address.csv", header=True, inferSchema=True)
city = spark.read.csv("city.csv", header=True, inferSchema=True)

In [6]:
movies_per_category = (
    film_category
    .join(category, "category_id")
    .groupBy("name")
    .agg(count("film_id").alias("movie_count"))
    .orderBy(desc("movie_count"))
)

print("1. Movies in each category")
movies_per_category.show(truncate=False)

1. Movies in each category
+-----------+-----------+
|name       |movie_count|
+-----------+-----------+
|Sports     |74         |
|Foreign    |73         |
|Family     |69         |
|Documentary|68         |
|Animation  |66         |
|Action     |64         |
|New        |63         |
|Drama      |62         |
|Games      |61         |
|Sci-Fi     |61         |
|Children   |60         |
|Comedy     |58         |
|Travel     |57         |
|Classics   |57         |
|Horror     |56         |
|Music      |51         |
+-----------+-----------+



In [7]:
top_actors = (
    rental
    .join(inventory, "inventory_id")
    .join(film_actor, "film_id")
    .join(actor, "actor_id")
    .groupBy("actor_id", "first_name", "last_name")
    .agg(count("rental_id").alias("total_rentals"))
    .orderBy(desc("total_rentals"))
    .limit(10)
)

print("2. Top 10 actors whose movies rented the most")
top_actors.show(truncate=False)

2. Top 10 actors whose movies rented the most
+--------+----------+-----------+-------------+
|actor_id|first_name|last_name  |total_rentals|
+--------+----------+-----------+-------------+
|107     |GINA      |DEGENERES  |753          |
|181     |MATTHEW   |CARREY     |678          |
|198     |MARY      |KEITEL     |674          |
|144     |ANGELA    |WITHERSPOON|654          |
|102     |WALTER    |TORN       |640          |
|60      |HENRY     |BERRY      |612          |
|150     |JAYNE     |NOLTE      |611          |
|37      |VAL       |BOLGER     |605          |
|23      |SANDRA    |KILMER     |604          |
|90      |SEAN      |GUINESS    |599          |
+--------+----------+-----------+-------------+



In [8]:
top_category_revenue = (
    payment
    .join(rental, "rental_id")
    .join(inventory, "inventory_id")
    .join(film_category, "film_id")
    .join(category, "category_id")
    .groupBy("name")
    .agg(round(sum("amount"), 2).alias("total_revenue"))
    .orderBy(desc("total_revenue"))
)

print("3. Category with highest revenue")
top_category_revenue.show(1, truncate=False)

3. Category with highest revenue
+------+-------------+
|name  |total_revenue|
+------+-------------+
|Sports|5314.21      |
+------+-------------+
only showing top 1 row


In [9]:
movies_not_in_inventory = (
    film
    .join(inventory, "film_id", "left_anti")
    .select("title")
)

print("4. Movies not in inventory")
movies_not_in_inventory.show(truncate=False)

4. Movies not in inventory
+----------------------+
|title                 |
+----------------------+
|ALICE FANTASIA        |
|APOLLO TEEN           |
|ARGONAUTS TOWN        |
|ARK RIDGEMONT         |
|ARSENIC INDEPENDENCE  |
|BOONDOCK BALLROOM     |
|BUTCH PANTHER         |
|CATCH AMISTAD         |
|CHINATOWN GLADIATOR   |
|CHOCOLATE DUCK        |
|COMMANDMENTS EXPRESS  |
|CROSSING DIVORCE      |
|CROWDS TELEMARK       |
|CRYSTAL BREAKING      |
|DAZED PUNK            |
|DELIVERANCE MULHOLLAND|
|FIREHOUSE VIETNAM     |
|FLOATS GARDEN         |
|FRANKENSTEIN STRANGER |
|GLADIATOR WESTWARD    |
+----------------------+
only showing top 20 rows


In [11]:
from pyspark.sql.functions import col, countDistinct, desc, dense_rank
from pyspark.sql.window import Window

children_actor_counts = (
    film_actor
    .join(film_category, "film_id")
    .join(category, "category_id")
    .filter(col("name") == "Children")
    .join(actor, "actor_id")
    .groupBy("actor_id", "first_name", "last_name")
    .agg(countDistinct("film_id").alias("movie_count"))
)

window_spec = Window.orderBy(desc("movie_count"))

ranked_children_actors = (
    children_actor_counts
    .withColumn("rank", dense_rank().over(window_spec))
    .filter(col("rank") <= 3)
    .orderBy("rank")
)

print("5. Top actors in Children category")
ranked_children_actors.show(truncate=False)

5. Top actors in Children category
+--------+----------+---------+-----------+----+
|actor_id|first_name|last_name|movie_count|rank|
+--------+----------+---------+-----------+----+
|17      |HELEN     |VOIGHT   |7          |1   |
|127     |KEVIN     |GARLAND  |5          |2   |
|80      |RALPH     |CRUZ     |5          |2   |
|66      |MARY      |TANDY    |5          |2   |
|140     |WHOOPI    |HURT     |5          |2   |
|81      |SCARLETT  |DAMON    |4          |3   |
|109     |SYLVESTER |DERN     |4          |3   |
|23      |SANDRA    |KILMER   |4          |3   |
|187     |RENEE     |BALL     |4          |3   |
|92      |KIRSTEN   |AKROYD   |4          |3   |
|173     |ALAN      |DREYFUSS |4          |3   |
|101     |SUSAN     |DAVIS    |4          |3   |
|150     |JAYNE     |NOLTE    |4          |3   |
|13      |UMA       |WOOD     |4          |3   |
|131     |JANE      |JACKMAN  |4          |3   |
|58      |CHRISTIAN |AKROYD   |4          |3   |
|142     |JADA      |RYDER    |4  

In [12]:
customer_city_status = (
    customer
    .join(address, "address_id")
    .join(city, "city_id")
    .groupBy("city")
    .agg(
        sum(when(col("active") == 1, 1).otherwise(0)).alias("active_customers"),
        sum(when(col("active") == 0, 1).otherwise(0)).alias("inactive_customers")
    )
    .orderBy(desc("inactive_customers"))
)

print("6. Cities with active/inactive customers")
customer_city_status.show(truncate=False)

6. Cities with active/inactive customers
+------------------+----------------+------------------+
|city              |active_customers|inactive_customers|
+------------------+----------------+------------------+
|Uluberia          |0               |1                 |
|Najafabad         |0               |1                 |
|Pingxiang         |0               |1                 |
|Xiangfan          |0               |1                 |
|Kumbakonam        |0               |1                 |
|Szkesfehrvr       |0               |1                 |
|Charlotte Amalie  |0               |1                 |
|Kamyin            |0               |1                 |
|Daxian            |0               |1                 |
|Coatzacoalcos     |0               |1                 |
|Wroclaw           |0               |1                 |
|Ktahya            |0               |1                 |
|Southend-on-Sea   |0               |1                 |
|Bat Yam           |0               |1         

In [13]:

rental_hours_corrected = (
    rental
    .withColumn(
        "rental_hours",
        (unix_timestamp(try_to_timestamp(col("return_date"), lit("yyyy-MM-dd HH:mm:ssX"))) - unix_timestamp(col("rental_date"))) / 3600
    )
)

city_category_hours_corrected = (
    rental_hours_corrected
    .join(customer, "customer_id")
    .join(address, "address_id")
    .join(city, "city_id")
    .join(inventory, "inventory_id")
    .join(film_category, "film_id")
    .join(category, "category_id")
    .groupBy("city", "name")
    .agg(round(sum("rental_hours"), 2).alias("total_hours"))
)


a_cities = (
    city_category_hours_corrected
    .filter(lower(col("city")).startswith("a"))
)

window_a = Window.partitionBy("city").orderBy(desc("total_hours"))

top_a_cities = (
    a_cities
    .withColumn("rank", dense_rank().over(window_a))
    .filter(col("rank") == 1)
    .select(
        col("city"),
        col("name").alias("category"),
        "total_hours"
    )
)


print('7A. Top category by rental hours for cities starting with "a"')
top_a_cities.show(truncate=False)

7A. Top category by rental hours for cities starting with "a"
+---------------------+--------+-----------+
|city                 |category|total_hours|
+---------------------+--------+-----------+
|A Corua (La Corua)   |Comedy  |592.75     |
|Abha                 |Sci-Fi  |550.43     |
|Abu Dhabi            |Sci-Fi  |495.32     |
|Acua                 |Drama   |552.55     |
|Adana                |Comedy  |533.17     |
|Addis Abeba          |Family  |529.1      |
|Aden                 |New     |795.23     |
|Adoni                |Children|409.18     |
|Ahmadnagar           |Children|587.93     |
|Akishima             |Children|652.65     |
|Akron                |Sports  |535.53     |
|Alessandria          |Comedy  |380.5      |
|Allappuzha (Alleppey)|New     |556.45     |
|Allende              |Travel  |685.82     |
|Almirante Brown      |Sports  |586.32     |
|Alvorada             |Sci-Fi  |447.78     |
|Ambattur             |Games   |499.3      |
|Amersfoort           |Sports  |583.93

In [14]:
dash_cities = (
    city_category_hours_corrected
    .filter(col("city").contains("-"))
)

window_dash = Window.partitionBy("city").orderBy(desc("total_hours"))

top_dash_cities = (
    dash_cities
    .withColumn("rank", dense_rank().over(window_dash))
    .filter(col("rank") == 1)
    .select(
        col("city"),
        col("name").alias("category"),
        "total_hours"
    )
)

print('7B. Top category by rental hours for cities containing "-"')
top_dash_cities.show(truncate=False)

7B. Top category by rental hours for cities containing "-"
+-----------------------+-----------+-----------+
|city                   |category   |total_hours|
+-----------------------+-----------+-----------+
|Augusta-Richmond County|Foreign    |526.12     |
|Beni-Mellal            |Foreign    |508.07     |
|Donostia-San Sebastin  |Sports     |800.38     |
|Effon-Alaiye           |Drama      |680.17     |
|Hubli-Dharwad          |Action     |290.53     |
|Jalib al-Shuyukh       |Classics   |615.3      |
|Jastrzebie-Zdrj        |Animation  |483.22     |
|Kamjanets-Podilskyi    |Children   |705.97     |
|Kirovo-Tepetsk         |Documentary|571.55     |
|Lapu-Lapu              |Comedy     |898.75     |
|Mwene-Ditu             |Travel     |572.6      |
|Naala-Porto            |Foreign    |485.35     |
|Saint-Denis            |Sci-Fi     |814.62     |
|Shahr-e Kord           |Foreign    |478.75     |
|Shubra al-Khayma       |Sci-Fi     |695.83     |
|Southend-on-Sea        |Sports     |504.

In [15]:
#Stop Session
spark.stop()